In [ ]:
import torch
from diffusers import AutoPipelineForInpainting
from diffusers.utils import load_image, make_image_grid

pipeline = AutoPipelineForInpainting.from_pretrained("kandinsky-community/kandinsky-2-2-decoder-inpaint",torch_dtype=torch.float16)
pipeline = pipeline.to("cuda")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recomme

model_index.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

model_index.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/776 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: /root/.cache/huggingface/hub/models--kandinsky-community--kandinsky-2-2-prior/snapshots/9fc51ad5732afc5d031724219d22e6c42179c5a8/image_encoder
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: /root/.cache/huggingface/hub/models--kandinsky-community--kandinsky-2-2-prior/snapshots/9fc51ad5732afc5d031724219d22e6c42179c5a8/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
#init_image = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint.png")
init_image = load_image("img_002.png")
#mask_image = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/inpaint_mask.png")
mask_image = load_image("mask_2.png")

prompt = "A set of keys lying amongst other office supplies on a table"
negative_prompt = "lurry, low quality, low resolution, deformed, distorted, cartoon, illustration, anime, painting, text, watermark, extra objects, duplicate objects, unrealistic lighting"

size = (431, 609)

init_image = init_image.resize(size)
mask_image = mask_image.resize(size)

image = pipeline(prompt=prompt, negative_prompt=negative_prompt, image=init_image, mask_image=mask_image,
    strength=0.95,               # A quel point il modifie l'image initial
    guidance_scale=4.9,          # A quel point on suit le prompt
    num_inference_steps=50).images[0]

image.save("resultat.png")
make_image_grid([init_image, mask_image, image], rows=1, cols=3)




NameError: name 'init_image' is not defined

In [ ]:
import os
import random
from PIL import Image, ImageEnhance, ImageFilter

NUM_IMAGES = 50

BACKGROUND_DIR = "back"     # Répertoire des backgrounds
KEYS_DIR = "keys"           # Répertoire des clés
OUTPUT_DIR = "generated_images"     # Répertoire des images générées

IMAGE_SIZE = (640, 640)

# taille des clés
MIN_KEY_SCALE = 0.04
MAX_KEY_SCALE = 0.15

# nombre de clés par image
MIN_KEYS_PER_IMAGE = 1
MAX_KEYS_PER_IMAGE = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)

background_files = [os.path.join(BACKGROUND_DIR, f) for f in os.listdir(BACKGROUND_DIR)]
key_files = [os.path.join(KEYS_DIR, f) for f in os.listdir(KEYS_DIR)]



def clean_alpha(img):               # pour enlever le fond blanc des clés
    img = img.convert("RGBA")
    datas = img.getdata()
    new_data = []
    for r, g, b, a in datas:
        # supprime faux fond blanc
        if r > 245 and g > 245 and b > 245:
            new_data.append((255, 255, 255, 0))
        else:
            new_data.append((r, g, b, a))
    img.putdata(new_data)
    return img

def transform_key(key, bg_width):

    # Changement de taille
    scale = random.uniform(MIN_KEY_SCALE, MAX_KEY_SCALE)
    target_width = int(bg_width * scale)
    ratio = target_width / key.width
    target_height = int(key.height * ratio)
    key = key.resize((target_width, target_height),Image.Resampling.LANCZOS)

    # Rotation
    angle = random.uniform(0, 360)
    key = key.rotate(angle,resample=Image.Resampling.BICUBIC,expand=True)

    # Clarté
    enhancer = ImageEnhance.Brightness(key)
    brightness = random.uniform(0.7, 1.3)
    key = enhancer.enhance(brightness)
    return key

def paste_key(background, key):    # Ajout de la clé dans l'image

    bg_w, bg_h = background.size
    key_w, key_h = key.size
    x = random.randint(0, max(0, bg_w - key_w))
    y = random.randint(0, max(0, bg_h - key_h))
    background.paste(key, (x, y), key)

# Création du dataset
for i in range(NUM_IMAGES):
    bg_path = random.choice(background_files)
    background = Image.open(bg_path).convert("RGBA")
    background = background.resize(IMAGE_SIZE)
    bg_w, bg_h = background.size
    num_keys = random.randint(MIN_KEYS_PER_IMAGE, MAX_KEYS_PER_IMAGE)

    for _ in range(num_keys):
        key_path = random.choice(key_files)
        key = Image.open(key_path).convert("RGBA")
        key = clean_alpha(key)
        key = transform_key(key, bg_w)
        paste_key(background, key)

    # Ajout d'un flou artistique
    if random.random() < 0.3:
        background = background.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))

    # SAVE
    output_path = os.path.join(OUTPUT_DIR, f"generated_{i:05d}.png")
    background.convert("RGB").save(output_path)

    if i % 100 == 0:
        print(f"Generated {i}/{NUM_IMAGES}")

print("DONE ✅")

Generated 0/50
DONE ✅
